In [3]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

In [4]:
#load dataset
data = pd.read_csv("../data/price_paid_records.csv")


In [5]:
#inital data overview
data.head()


,Transaction unique identifier,Price,Date of Transfer,Property Type,Old/New,Duration,Town/City,District,County,PPDCategory Type,Record Status - monthly file only
0,{81B82214-7FBC-4129-9F6B-4956B4A663AD},25000,1995-08-18 00:00,T,N,F,OLDHAM,OLDHAM,GREATER MANCHESTER,A,A
1,{8046EC72-1466-42D6-A753-4956BF7CD8A2},42500,1995-08-09 00:00,S,N,F,GRAYS,THURROCK,THURROCK,A,A
2,{278D581A-5BF3-4FCE-AF62-4956D87691E6},45000,1995-06-30 00:00,T,N,F,HIGHBRIDGE,SEDGEMOOR,SOMERSET,A,A
3,{1D861C06-A416-4865-973C-4956DB12CD12},43150,1995-11-24 00:00,T,N,F,BEDFORD,NORTH BEDFORDSHIRE,BEDFORDSHIRE,A,A
4,{DD8645FD-A815-43A6-A7BA-4956E58F1874},18899,1995-06-23 00:00,S,N,F,WAKEFIELD,LEEDS,WEST YORKSHIRE,A,A


In [6]:
data.shape

(22489348, 11)

In [7]:
data.columns

Index(['Transaction unique identifier', 'Price', 'Date of Transfer',
       'Property Type', 'Old/New', 'Duration', 'Town/City', 'District',
       'County', 'PPDCategory Type', 'Record Status - monthly file only'],
      dtype='object')

In [8]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22489348 entries, 0 to 22489347
Data columns (total 11 columns):
 #   Column                             Dtype 
---  ------                             ----- 
 0   Transaction unique identifier      object
 1   Price                              int64 
 2   Date of Transfer                   object
 3   Property Type                      object
 4   Old/New                            object
 5   Duration                           object
 6   Town/City                          object
 7   District                           object
 8   County                             object
 9   PPDCategory Type                   object
 10  Record Status - monthly file only  object
dtypes: int64(1), object(10)
memory usage: 1.8+ GB


In [9]:
data.describe()

,Price
count,2.248935e+07
mean,1.782442e+05
std,3.903677e+05
min,1.000000e+00
25%,7.500000e+04
50%,1.300000e+05
75%,2.100000e+05
max,9.890000e+07


In [10]:
data.isnull().sum()

Transaction unique identifier        0
Price                                0
Date of Transfer                     0
Property Type                        0
Old/New                              0
Duration                             0
Town/City                            0
District                             0
County                               0
PPDCategory Type                     0
Record Status - monthly file only    0
dtype: int64

In [11]:
#date column conversion
data["Date of Transfer"] = pd.to_datetime(data["Date of Transfer"])
data["Year"] = data["Date of Transfer"].dt.year

In [12]:
#remove unnecessary columns
data = data.drop(columns=[
    "Transaction unique identifier",
    "PPDCategory Type",
    "Record Status - monthly file only"
])

In [13]:
#updated dataset
data.head()

,Price,Date of Transfer,Property Type,Old/New,Duration,Town/City,District,County,Year
0,25000,1995-08-18,T,N,F,OLDHAM,OLDHAM,GREATER MANCHESTER,1995
1,42500,1995-08-09,S,N,F,GRAYS,THURROCK,THURROCK,1995
2,45000,1995-06-30,T,N,F,HIGHBRIDGE,SEDGEMOOR,SOMERSET,1995
3,43150,1995-11-24,T,N,F,BEDFORD,NORTH BEDFORDSHIRE,BEDFORDSHIRE,1995
4,18899,1995-06-23,S,N,F,WAKEFIELD,LEEDS,WEST YORKSHIRE,1995


In [14]:
#remove missing values
data.isnull().sum()

Price               0
Date of Transfer    0
Property Type       0
Old/New             0
Duration            0
Town/City           0
District            0
County              0
Year                0
dtype: int64

In [15]:
#If there are missing rows, remove them.
data = data.dropna()
data.isnull().sum()

Price               0
Date of Transfer    0
Property Type       0
Old/New             0
Duration            0
Town/City           0
District            0
County              0
Year                0
dtype: int64

In [27]:
#Create encoder
property_type_encoder = LabelEncoder()
old_new_encoder = LabelEncoder()
duration_encoder = LabelEncoder()
town_city_encoder = LabelEncoder()
district_encoder = LabelEncoder()
county_encoder = LabelEncoder()

data["Property Type"] = property_type_encoder.fit_transform(data["Property Type"])
data["Old/New"] = old_new_encoder.fit_transform(data["Old/New"])
data["Duration"] = duration_encoder.fit_transform(data["Duration"])
data["Town/City"] = town_city_encoder.fit_transform(data["Town/City"])
data["District"] = district_encoder.fit_transform(data["District"])
data["County"] = county_encoder.fit_transform(data["County"])

In [28]:
joblib.dump(property_type_encoder, "../models/property_type_encoder.pkl")
joblib.dump(old_new_encoder, "../models/old_new_encoder.pkl")
joblib.dump(duration_encoder, "../models/duration_encoder.pkl")
joblib.dump(town_city_encoder, "../models/town_city_encoder.pkl")
joblib.dump(district_encoder, "../models/district_encoder.pkl")
joblib.dump(county_encoder, "../models/county_encoder.pkl")

['../models/county_encoder.pkl']

In [18]:
#Define Features and Target
y = data["Price"]
X = data.drop(columns=["Price", "Date of Transfer"])
X.head()

,Property Type,Old/New,Duration,Town/City,District,County,Year
0,4,0,0,748,279,47,1995
1,3,0,0,421,397,109,1995
2,4,0,0,479,327,94,1995
3,4,0,0,84,255,3,1995
4,3,0,0,1064,212,119,1995


In [19]:
#Split the Dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [20]:
#Check dataset sizes
print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (17991478, 7)
Testing data: (4497870, 7)


In [21]:
#Train Linear Regression Model

#Create the model
lr_model = LinearRegression()

#Train the model
lr_model.fit(X_train, y_train)
joblib.dump(lr_model, "../models/linear_regression_model.pkl")

#Make predictions
lr_predictions = lr_model.predict(X_test)

In [22]:
#Train Decision Tree Model

#Create the model
dt_model = DecisionTreeRegressor(random_state=42)

#Train
dt_model.fit(X_train, y_train)
joblib.dump(dt_model, "../models/decision_tree_model.pkl")
#Predict
dt_predictions = dt_model.predict(X_test)

In [23]:
#Train Random Forest Model

#Create model
rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

#Train model
rf_model.fit(X_train, y_train)
joblib.dump(rf_model, "../models/random_forest_model.pkl")

#Predict
rf_predictions = rf_model.predict(X_test)

In [24]:
#Evaluate Model Performance

#Linear Regression Evaluation

print("Linear Regression")

print("MAE:", mean_absolute_error(y_test, lr_predictions))
print("MSE:", mean_squared_error(y_test, lr_predictions))
print("R2 Score:", r2_score(y_test, lr_predictions))

Linear Regression
MAE: 90975.77975484704
MSE: 143007515215.44272
R2 Score: 0.05253275164537585


In [25]:
#Decision Tree Evaluation
print("Decision Tree")

print("MAE:", mean_absolute_error(y_test, dt_predictions))
print("MSE:", mean_squared_error(y_test, dt_predictions))
print("R2 Score:", r2_score(y_test, dt_predictions))

Decision Tree
MAE: 57533.801145526755
MSE: 112754394880.33965
R2 Score: 0.2529686562540181


In [26]:
#Random Forest Evaluation
print("Random Forest")

print("MAE:", mean_absolute_error(y_test, rf_predictions))
print("MSE:", mean_squared_error(y_test, rf_predictions))
print("R2 Score:", r2_score(y_test, rf_predictions))

Random Forest
MAE: 57431.869850244875
MSE: 110951283893.83551
R2 Score: 0.2649148018973946


In [29]:
#best model
joblib.dump(rf_model, "../models/house_price_model.pkl")

['../models/house_price_model.pkl']